# Reto Vientos

Dos pasos separados:

| Paso | Qué hace | Red |
|---|---|---|
| **1 · Crawling** | Descarga con crawl4ai el viento de todas las estaciones | sí |
| **2 · Datos** | Revisa las lecturas en tablas, las guarda en `retos/output/vientos/<fecha>/` y comprueba que los ficheros están completos | no |

| Fuente | Qué es | Clave |
|---|---|---|
| AEMET OpenData | ~855 estaciones automáticas (observación) | AEMET |
| Puertos del Estado (iMar / PORTUS) | ~100 boyas, mareógrafos y estaciones de puerto (observación) | no |
| Open-Meteo | 52 capitales de provincia (modelo, respaldo) | no |

Cada lectura incluye:
- velocidad y racha (km/h);
- de dónde viene (`direccion_grados`) y hacia dónde empuja (`hacia_grados`);
- temperatura y humedad;
- `regla_30` (>30 °C, <30 % de humedad y >30 km/h);
- `antiguedad_min`: minutos desde la medida. No se descarta ninguna estación por ir atrasada.

Solo produce datos, no abre ningún mapa: para verlos en el mapa interactivo está el pipeline completo (`pipeline_incendios.ipynb`).

## 0. Preparación

In [1]:
# Recarga automática: si el código de incendios/ o retos/ cambia, se usa la versión nueva sin reiniciar.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "incendios" / "__init__.py").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No encuentro la carpeta incendios/: abre el notebook desde Incendios-V0.0/ o retos/")
for ruta in (RAIZ, RAIZ / "retos"):
    if str(ruta) not in sys.path:
        sys.path.insert(0, str(ruta))

import pandas as pd

from incendios.config import fijar_claves, estado_claves
from incendios.pipeline import resumen
from reto_vientos import comprobar, crawlear, guardar, leer_json, nueva_carpeta

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("Proyecto:", RAIZ)

Proyecto: /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0


## 1. Claves

- **AEMET:** necesaria para las ~855 estaciones de AEMET.
- **Sin ella:** Puertos del Estado y Open-Meteo funcionan igual.
- **LSA-SAF:** no se usa en este reto.

Si ya están en `.env` (por ejemplo, porque las guardaste desde el pipeline), deja las comillas vacías. ⚠️ Si las pegas aquí, no subas este notebook a git.

In [2]:
AEMET_API_KEY   = ""
LSASAF_USUARIO  = ""   # opcional (SEVIRI)
LSASAF_PASSWORD = ""   # opcional (SEVIRI)

fijar_claves(aemet=AEMET_API_KEY, lsasaf_usuario=LSASAF_USUARIO, lsasaf_password=LSASAF_PASSWORD)
for variable, valor in estado_claves().items():
    print(f"{variable:<16} {valor}")

AEMET_API_KEY    eyJh…EFOA (284 caracteres)
FIRMS_MAP_KEY    — (vacía)
LSASAF_USER      ivan…te19 (12 caracteres)
LSASAF_PASSWORD  ****


# PASO 1 · Crawling

Las tres fuentes se lanzan en paralelo con crawl4ai. Cada una va aislada: si falla o le falta la clave, queda anotada y las demás siguen.

### 1.1 Ejecutar el crawling

In [3]:
resultado = await crawlear()
print(resumen(resultado["estado"]))
df = pd.DataFrame(resultado["lecturas"])
display(df.groupby(["fuente", "tipo_dato"]).size().rename("estaciones").to_frame())

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

[INIT].... → Crawl4AI 0.8.9 

  ok       Open-Meteo (capitales)               52 registros    0.3s
  ok       AEMET (estaciones)                  854 registros    2.1s
  ok       Puertos del Estado (costa)          104 registros    3.6s


,,estaciones
fuente,tipo_dato,
AEMET,observación,854
Open-Meteo,modelo,52
Puertos del Estado,observación,104


# PASO 2 · Datos

Revisión de las lecturas, guardado en ficheros (`viento.geojson`, `estado.json` y `manifiesto.json`) y comprobación de que se han escrito completos. No hace peticiones de red: trabaja con lo que ha descargado el paso 1.

### 2.1 Rachas más fuertes

In [4]:
columnas = ["fuente", "nombre", "antiguedad_min", "velocidad_kmh", "racha_kmh", "direccion_cardinal", "temperatura_c", "humedad_pct", "regla_30"]
display(df.sort_values("racha_kmh", ascending=False)[columnas].head(15))

,fuente,nombre,antiguedad_min,velocidad_kmh,racha_kmh,direccion_cardinal,temperatura_c,humedad_pct,regla_30
87,AEMET,VEGA DE URRIELLU · PARQUE NACIONAL PICOS DE EU...,57,31.0,67.0,NE,5.9,93.0,False
78,AEMET,TORRELAVEGA-SIERRAPANDO,57,31.7,64.8,O,16.1,98.0,False
74,AEMET,SANTANDER CMT,57,29.5,63.0,ONO,18.0,96.0,False
822,AEMET,LA ALDEA DE SAN NICOLAS,57,18.0,60.8,NE,28.2,54.0,False
69,AEMET,CASTRO URDIALES-EDAR,57,27.7,58.7,OSO,18.0,87.0,False
750,AEMET,PORTOCOLOM,57,5.4,58.3,SSE,21.2,81.0,False
769,AEMET,MENORCA/AEROPUERTO,57,23.8,55.4,OSO,24.2,58.0,False
751,AEMET,SON SERVERA-CAN PEP MONJO,117,9.7,54.7,ONO,19.7,93.0,False
73,AEMET,SANTANDER AEROPUERTO,57,33.5,53.6,O,16.9,95.0,False
935,Puertos del Estado,Estac. Meteo. Sagunto Norte,7,16.6,52.6,SSO,29.5,NaN,None


### 2.2 Condiciones de riesgo

Estaciones con la humedad más baja y las que cumplen la regla del 30, el umbral de condiciones extremas de propagación.

In [5]:
print("Estaciones en regla del 30:", int(df["regla_30"].fillna(False).sum()))
display(df.sort_values("humedad_pct")[columnas].head(10))

Estaciones en regla del 30: 0


,fuente,nombre,antiguedad_min,velocidad_kmh,racha_kmh,direccion_cardinal,temperatura_c,humedad_pct,regla_30
474,AEMET,ESTEPONA,57,23.0,37.1,OSO,29.7,29.0,False
801,AEMET,CAÑADAS PARADOR,117,17.3,33.8,O,18.0,33.0,False
511,AEMET,HUERCAL OVERA,57,19.1,32.0,ONO,27.9,35.0,False
552,AEMET,ALICANTE,57,10.4,26.6,ONO,29.2,36.0,False
978,Open-Meteo,Alicante,12,12.3,26.6,O,29.2,36.0,False
487,AEMET,VELEZ MALAGA (AUTOMATICA),57,15.8,28.8,ONO,29.5,36.0,False
483,AEMET,MÁLAGA/AEROPUERTO,57,15.1,31.7,ONO,28.0,36.0,False
977,Open-Meteo,Valencia,12,14.4,32.8,O,28.7,37.0,False
999,Open-Meteo,Sevilla,12,9.0,23.8,ONO,27.9,37.0,False
820,AEMET,TEJEDA CASCO,57,7.2,22.7,OSO,25.5,38.0,False


### 2.3 Antigüedad de las lecturas

No se descarta ninguna estación por ir atrasada; aquí se ve cuántas tienen una lectura reciente.

In [6]:
tramos = pd.cut(df["antiguedad_min"], [-1, 30, 90, 180, 360, float("inf")],
                labels=["≤ 30 min", "30–90 min", "1,5–3 h", "3–6 h", "> 6 h"])
display(tramos.value_counts().sort_index().rename("estaciones").to_frame())

,estaciones
antiguedad_min,
≤ 30 min,130
30–90 min,581
"1,5–3 h",237
3–6 h,59
> 6 h,3


### 2.4 Guardar los ficheros

Escribe `viento.geojson`, `estado.json` y `manifiesto.json` en una carpeta nueva. El manifiesto indica qué fichero es cada capa y cuántos registros tiene, para que cualquier otra aplicación pueda leerlos.

In [7]:
carpeta = nueva_carpeta("vientos")
guardar(resultado, carpeta)
print("Guardado en", carpeta)
manifiesto = leer_json(carpeta / "manifiesto.json")
print(manifiesto["capas"]["viento"]["por_fuente"])

Guardado en /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/retos/output/vientos/20261007T135634Z
{'AEMET': 854, 'Puertos del Estado': 104, 'Open-Meteo': 52}


### 2.5 Comprobar los ficheros guardados

Se relee lo que se acaba de escribir y se compara con el manifiesto y con lo que hay en memoria: cada fichero tiene que existir, poder leerse y tener todos sus registros.
- **Todo ✓:** los datos están completos.
- **Algún ✗:** vuelve a ejecutar la sección anterior.

In [8]:
tabla = pd.DataFrame(comprobar(carpeta))
tabla["ok"] = tabla["ok"].map({True: "✓", False: "✗"})
display(tabla)
print("Datos completos ✓" if (tabla["ok"] == "✓").all() else "Hay ficheros incompletos ✗: vuelve a guardar")
print()
print("Carpeta de datos:", carpeta)
for f in sorted(carpeta.rglob("*")):
    if f.is_file():
        print(f"   {f.relative_to(carpeta)}  ({f.stat().st_size // 1024} KB)")

,capa,fichero,manifiesto,en el fichero,ok
0,viento,viento.geojson,1010,1010,✓


Datos completos ✓

Carpeta de datos: /Users/ivanduartepacheco/Downloads/trabajo/proyecto/Incendios-V0.0/retos/output/vientos/20261007T135634Z
   estado.json  (0 KB)
   manifiesto.json  (0 KB)
   viento.geojson  (459 KB)


### Mapa interactivo

Genera `mapa_vientos.html` en la carpeta de datos y lo abre en el navegador. Lee solo los ficheros guardados: no hace peticiones.

In [ ]:
from reto_vientos import mapa
ruta = mapa(carpeta, abrir=True)
print("Mapa:", ruta)